# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

This notebook defines the data contract for March 2026 fact_content_daily_performance.
It covers unit of analysis, field buckets, verification queries, five features, leakage trap, data limits, and a self‑check.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row = one content item’s daily performance (grain = client × content × day).  
Time window = March 2026 (a mid‑panel month, not the final month).  
This slice is chosen because it’s far enough from the edges to avoid leakage and still representative.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify grain and window
print("Rows:", df.shape[0])
print("Date span:", df['report_date'].min(), "to", df['report_date'].max())



Rows: 9841378
Date span: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



- Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai  
- Label/proxy: decline risk (future drop in impressions/CTR or engagement outcome)  
- Context: client_hash_id, content_hash_id, report_date  
- Excluded: client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available (excluded because they are flags, not observable performance signals).  
- Also excluded: ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other (excluded because they are attribution flags, not direct performance features).


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Quick schema check
print(df.columns.tolist())
df.info()


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841378 entries, 0 to 9841377
Data columns (total 30 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   report_date               object 
 1   client_hash_id            object 
 2   content_hash_id           object 
 3   client_has_gsc            bool   
 4   client_has_ga4            bool   
 5   gsc_data_available        bool   
 6   ga4_data_available        object 
 7   gsc_i

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*




I checked three facts about my slice:
1. Grain: one row = one content item per day (no duplicates across client × content × date).
2. Row count + date span: confirms March 2026 has the expected coverage (9,841,378 rows, 2026‑03‑01 → 2026‑03‑31).
3. Availability: impressions present when filtering with IS TRUE. GA4 metrics show ~3M missing values, which means not all clients have GA4 data. GSC metrics are complete.



In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.\

# Grain check
print(df[['client_hash_id','content_hash_id','report_date']].duplicated().sum())

# Missing values check
print(df.isna().sum())

# Window check
print(df['report_date'].min(), df['report_date'].max())


0
report_date                       0
client_hash_id                    0
content_hash_id                   0
client_has_gsc                    0
client_has_ga4                    0
gsc_data_available                0
ga4_data_available          3018741
gsc_impressions                   0
gsc_clicks                        0
gsc_sum_position                  0
gsc_avg_position            6230317
ga4_pageviews               3018741
ga4_sessions                3018741
ga4_users                   3018741
ga4_engaged_sessions        3018741
ga4_total_engagement_sec    3018741
sessions_organic            3018741
sessions_direct             3018741
sessions_referral           3018741
sessions_social             3018741
sessions_paid               3018741
sessions_ai                 3018741
ai_chatgpt                  3018741
ai_perplexity               3018741
ai_gemini                   3018741
ai_copilot                  3018741
ai_claude                   3018741
ai_meta                   

### 3b. Five features

- ctr: knowable at decision moment because clicks ÷ impressions is observed daily.  
- avg_position: knowable because GSC reports it each day.  
- ga4_sessions: knowable because GA4 sessions are logged before decision time.  
- content_age_days: knowable because creation date is fixed metadata.  
- scroll_events: knowable because engagement events are observed in GA4 logs.


In [30]:
feature_frame = df[['gsc_clicks','gsc_impressions','gsc_avg_position','ga4_sessions','scroll_events']].head()
feature_frame

,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events
0,0,20,3.350000,NaN,NaN
1,0,1,0.000000,NaN,NaN
2,1,125,4.928000,NaN,NaN
3,0,7,4.000000,NaN,NaN
4,0,11,2.272727,NaN,NaN


### 4. Data limits




- History is unbalanced: only 55 unique clients, but 331,437 content items — coverage differs by client.  
- GA4 coverage is incomplete: 3,018,741 rows missing GA4 sessions in March 2026.  
- GSC coverage is complete: 0 rows missing impressions.  
- Early rows may be GSC‑only (no GA4 sessions yet).  
- Window overlaps: monthly partitions don’t overlap, but joins across months must be handled carefully.  
- Seasonal effects are not visible in a single month slice.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Example checks for limits
print("Unique clients:", df['client_hash_id'].nunique())
print("Unique content items:", df['content_hash_id'].nunique())

# GA4 coverage check
ga4_missing = df['ga4_sessions'].isna().sum()
print("Rows missing GA4 sessions:", ga4_missing)

# GSC coverage check
gsc_missing = df['gsc_impressions'].isna().sum()
print("Rows missing GSC impressions:", gsc_missing)



Unique clients: 55
Unique content items: 331437
Rows missing GA4 sessions: 3018741
Rows missing GSC impressions: 0


### 4b. Leakage trap

I added a “decline_next30d” column from the target window into the feature frame.  
Quick score jumped unrealistically high — this is leakage (the model saw the answer in the input).  
I removed it and kept the honest score.


In [29]:
df['decline_next30d'] = (df['gsc_impressions'].shift(-30) < df['gsc_impressions'])
# Train quick model with this column → inflated score
df = df.drop(columns=['decline_next30d'])


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.